# Imports

In [1]:
import os
import dotenv
import re
import pandas as pd
from googleapiclient.discovery import build

# Data Acquisition

In [2]:
# Load environment variables
dotenv.load_dotenv()

True

In [3]:
# Get the API key 
API_KEY = os.getenv("API_KEY")

# Set up the YouTube Data API client 
youtube = build("youtube", "v3", developerKey=API_KEY)

In [4]:
# Define the channel ID
SKZ_CHANNEL_ID = "UC9rMiEjNaCSsebs31MRDCRA"

# Make a request to get the 'uploads' playlist ID later
request = youtube.channels().list(
    # 'part="contentDetails"' tells YouTube we only want the structural info, saving quota
    part="contentDetails",
    id=SKZ_CHANNEL_ID
    )
response = request.execute()
display(response)

{'kind': 'youtube#channelListResponse',
 'etag': 'dfngdAEoaq4PHM-4mOx7L9ps-OM',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': 'GvS6bYaVU5IyMuZIilfP0HTnIdo',
   'id': 'UC9rMiEjNaCSsebs31MRDCRA',
   'contentDetails': {'relatedPlaylists': {'likes': '',
     'uploads': 'UU9rMiEjNaCSsebs31MRDCRA'}}}]}

In [22]:
# Define the channel ID
SKZ_CHANNEL_ID = "UC9rMiEjNaCSsebs31MRDCRA"

# Map the base Channel ID to the specific hidden playlists
base_id = SKZ_CHANNEL_ID.replace("UC", "")
playlists_to_process = {
    'Long-form': f'UULF{base_id}',
    'Short': f'UUSH{base_id}',
    'Live/VOD': f'UULV{base_id}'
}

In [5]:
# Find the 'uploads' playlist ID
uploads_playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
print(f"Stray Kids Uploads Playlist ID: {uploads_playlist_id}")

Stray Kids Uploads Playlist ID: UU9rMiEjNaCSsebs31MRDCRA


In [6]:
# Fetch video IDs via Pagination
# For testing, limit this to just 2 pages (2 x 50 videos)
video_ids = []
next_page_token = None

response_1 = youtube.playlistItems().list(
    part="contentDetails",
    playlistId=uploads_playlist_id,
    maxResults=50,
    pageToken=next_page_token
).execute()
display(response_1)

{'kind': 'youtube#playlistItemListResponse',
 'etag': 'wAalCVRg93qgbNd-1R4nxh-nSn0',
 'nextPageToken': 'EAAaHlBUOkNESWlFRU5DTTBVelJVUTRORGM1TlRRNFF6Zw',
 'items': [{'kind': 'youtube#playlistItem',
   'etag': 'uK-JWiJTRMchgPOJd0Ta3UB8-0o',
   'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLnBYUmVwZzZ1U1Nj',
   'contentDetails': {'videoId': 'pXRepg6uSSc',
    'videoPublishedAt': '2026-03-24T02:59:58Z'}},
  {'kind': 'youtube#playlistItem',
   'etag': '4B3ObKVmsgjsxb7r_fEPEWMcJ7g',
   'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLlhMQkRnNlBvWURZ',
   'contentDetails': {'videoId': 'XLBDg6PoYDY',
    'videoPublishedAt': '2026-03-20T09:51:36Z'}},
  {'kind': 'youtube#playlistItem',
   'etag': 'hVr8BfxoBqntZnFxsZdt9TwQKN8',
   'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLlFMZUdlMWV3dlNR',
   'contentDetails': {'videoId': 'QLeGe1ewvSQ',
    'videoPublishedAt': '2026-03-20T09:10:46Z'}},
  {'kind': 'youtube#playlistItem',
   'etag': 's9mlSvTI07UNGCEY5mlj2bJyxto',
   'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLnVGOVdXeDB

In [7]:
# Get the list of playllist items
print(len(response_1["items"]), " values")
display(response_1["items"])

50  values


[{'kind': 'youtube#playlistItem',
  'etag': 'uK-JWiJTRMchgPOJd0Ta3UB8-0o',
  'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLnBYUmVwZzZ1U1Nj',
  'contentDetails': {'videoId': 'pXRepg6uSSc',
   'videoPublishedAt': '2026-03-24T02:59:58Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': '4B3ObKVmsgjsxb7r_fEPEWMcJ7g',
  'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLlhMQkRnNlBvWURZ',
  'contentDetails': {'videoId': 'XLBDg6PoYDY',
   'videoPublishedAt': '2026-03-20T09:51:36Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': 'hVr8BfxoBqntZnFxsZdt9TwQKN8',
  'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLlFMZUdlMWV3dlNR',
  'contentDetails': {'videoId': 'QLeGe1ewvSQ',
   'videoPublishedAt': '2026-03-20T09:10:46Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': 's9mlSvTI07UNGCEY5mlj2bJyxto',
  'id': 'VVU5ck1pRWpOYUNTc2ViczMxTVJEQ1JBLnVGOVdXeDBlMnQw',
  'contentDetails': {'videoId': 'uF9WWx0e2t0',
   'videoPublishedAt': '2026-03-20T05:59:16Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': 'XsoOFS4VNt2BqpthrMAH87OPPUQ',
 

In [8]:
response_1["items"][0]["contentDetails"]["videoId"]

'pXRepg6uSSc'

In [9]:
[item["contentDetails"] for item in response_1["items"]]

[{'videoId': 'pXRepg6uSSc', 'videoPublishedAt': '2026-03-24T02:59:58Z'},
 {'videoId': 'XLBDg6PoYDY', 'videoPublishedAt': '2026-03-20T09:51:36Z'},
 {'videoId': 'QLeGe1ewvSQ', 'videoPublishedAt': '2026-03-20T09:10:46Z'},
 {'videoId': 'uF9WWx0e2t0', 'videoPublishedAt': '2026-03-20T05:59:16Z'},
 {'videoId': 'l8sSQAa1kNM', 'videoPublishedAt': '2026-03-19T15:00:08Z'},
 {'videoId': '9DSOFyLSv6Q', 'videoPublishedAt': '2026-03-19T10:59:58Z'},
 {'videoId': 'Jl5nNqajrV0', 'videoPublishedAt': '2026-03-18T10:59:56Z'},
 {'videoId': '1tipk2X1aUo', 'videoPublishedAt': '2026-03-17T10:59:56Z'},
 {'videoId': 'jePtbHuKQkM', 'videoPublishedAt': '2026-03-16T04:58:59Z'},
 {'videoId': 'bDbzZ_Dx4Ks', 'videoPublishedAt': '2026-03-15T11:00:18Z'},
 {'videoId': '18HxBEae2rc', 'videoPublishedAt': '2026-03-13T07:01:51Z'},
 {'videoId': 'uB-XPe1K7SU', 'videoPublishedAt': '2026-03-12T10:59:56Z'},
 {'videoId': 'X7jAms_f4Rc', 'videoPublishedAt': '2026-03-10T11:00:03Z'},
 {'videoId': 'bSZL3ZZC8EE', 'videoPublishedAt': '20

In [10]:
# Append the first 50 video IDs to video_ids
vid_ids_1 = [item["contentDetails"]["videoId"] for item in response_1["items"]]
video_ids.extend(vid_ids_1)
display(video_ids)

['pXRepg6uSSc',
 'XLBDg6PoYDY',
 'QLeGe1ewvSQ',
 'uF9WWx0e2t0',
 'l8sSQAa1kNM',
 '9DSOFyLSv6Q',
 'Jl5nNqajrV0',
 '1tipk2X1aUo',
 'jePtbHuKQkM',
 'bDbzZ_Dx4Ks',
 '18HxBEae2rc',
 'uB-XPe1K7SU',
 'X7jAms_f4Rc',
 'bSZL3ZZC8EE',
 'o66KJX1YJ3s',
 'ZV5IjNgpBFQ',
 'sKi1gEtnSAQ',
 'x-kl7fzEmaY',
 'lypgZaTn-Mg',
 'h1li3NlTB0I',
 'kcGUSYNoLmE',
 'SLhVEFCbN3s',
 'wtAtZfGt0qU',
 't4TGvBI8RN8',
 'XezXjA_V184',
 'ODoEa_1gPSc',
 'Ax76YDZ3qns',
 'HFbirADU3_Q',
 'GOmL3SaOzn8',
 'RQDj6OIb6ik',
 '7y6tpERbHO4',
 'dYHqCr0GCgA',
 '1SOy5r-vHSA',
 'RboGy-EZ4qs',
 'FEDQ14t58CA',
 'X1Ff6AZLcrA',
 '740hHN3iZLU',
 'cQ2CM4HOnB4',
 'mUFKUJTMjWg',
 'w_XIdjXgsjE',
 'jNX-Ix1T28Q',
 'PUUxE7EGP4Q',
 'TuT9SdQIJ44',
 '1v-Q_emAeN8',
 'r_HsUx72XtU',
 'OSUHqAvuSRI',
 'F6lf26nWk10',
 'Tsr3lBEn9ww',
 'IiqoozFOs1s',
 'v1xm6Pck-mo']

In [11]:
# Fetch the last 50 video IDs
next_page_token = response_1.get("nextPageToken")

response_2 = youtube.playlistItems().list(
    part="contentDetails",
    playlistId=uploads_playlist_id,
    maxResults=50,
    pageToken=next_page_token
).execute()

vid_ids_2 = [item["contentDetails"]["videoId"] for item in response_2["items"]]
video_ids.extend(vid_ids_2)

In [12]:
# Total of the video IDs collected so far
len(video_ids)

100

In [ ]:
# Create A Function to Categorize The Content using Regex
def categorize_content(title: str) -> str:
    title_upper = title.upper()
    
    if re.search(r'SKZ CODE|SKZCODE', title_upper):
        return 'SKZ CODE'
    elif re.search(r'2 KIDS ROOM|TWO KIDS ROOM', title_upper):
        return '2 Kids Room'
    elif re.search(r'1 KIDS ROOM', title_upper):
        return '1 Kids Room'
    elif re.search(r'SKZ-TALKER|SKZ TALKER', title_upper):
        return 'SKZ-TALKER'
    elif re.search(r'SKZ-PLAYER|SKZ-RECORD', title_upper):
        return 'SKZ-PLAYER/RECORD'
    elif re.search(r'UNVEIL : TRACK|M/V TEASER|TRAILER', title_upper):
        return 'Teaser/Trailer'
    elif re.search(r'M/V$', title_upper) or re.search(r'MUSIC VIDEO', title_upper):
        return 'Official M/V'
    elif re.search(r'VLOG', title_upper):
        return 'Vlog'
    else:
        return 'Other/Misc'

In [14]:
# Test The Function
test_titles = ["Stray Kids 'S-Class' M/V", "[SKZ CODE] Ep.30", "Just a random live"]
for t in test_titles:
    print(f"'{t}'  -->  {categorize_content(t)}")

'Stray Kids 'S-Class' M/V'  -->  Official M/V
'[SKZ CODE] Ep.30'  -->  SKZ CODE
'Just a random live'  -->  Other/Misc


In [15]:
video_ids[0:2]

['pXRepg6uSSc', 'XLBDg6PoYDY']

In [16]:
# Extract Stats for One Video
vid_response = youtube.videos().list(
    part="snippet,statistics",
    id=video_ids[0]
).execute()
display(vid_response)

{'kind': 'youtube#videoListResponse',
 'etag': 'KIMb7N9zk6XvJZbZcPhzU6IC8Nw',
 'items': [{'kind': 'youtube#video',
   'etag': 'UNxGYEZIf0ZFRn0L56vjr9sHwOQ',
   'id': 'pXRepg6uSSc',
   'snippet': {'publishedAt': '2026-03-24T02:59:58Z',
    'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
    'title': '빵발란스한 #StrayKids #스트레이키즈 #아이엔 #I_N #YouMakeStrayKidsStay',
    'description': '빵발란스한 #StrayKids #스트레이키즈 #아이엔 #I_N #YouMakeStrayKidsStay\n\nListen to "DO IT" now☁️\nhttps://Stray-Kids.lnk.to/DOIT\n\nListen to "Do It (Remixes)" now🎊\nhttps://Stray-Kids.lnk.to/DoItRemixes\n\nStray Kids "DO IT"\niTunes & Apple Music: https://Stray-Kids.lnk.to/DOIT/AppleMusic\nSpotify: https://Stray-Kids.lnk.to/DOIT/Spotify\n\nStray Kids "Do It (Remixes)"\niTunes & Apple Music: https://Stray-Kids.lnk.to/DoItRemixes/AppleMusic\nSpotify: https://Stray-Kids.lnk.to/DoItRemixes/Spotify\n\nStray Kids Official YouTube: https://www.youtube.com/c/StrayKids\nStray Kids Official Twitter: https://twitter.com/Stray_Kids\nStray Kids

In [ ]:
# Get the Video Stats
vid_info = vid_response['items'][0]
title = vid_info['snippet']['title']
stats = vid_info['statistics']

print(f"Title: {title}")
print(f"Views: {stats.get('viewCount')}")
print(f"Category: {categorize_content(title)}")

Title: 빵발란스한 #StrayKids #스트레이키즈 #아이엔 #I_N #YouMakeStrayKidsStay
Views: 899261
Category: Other/Misc


In [19]:
# Extract Top Comments for One Video
comment_response = youtube.commentThreads().list(
        part="snippet",
        videoId=video_ids[0],
        maxResults=5, # Just 5 for testing
        order="relevance", 
        textFormat="plainText"
    ).execute()
display(comment_response)

{'kind': 'youtube#commentThreadListResponse',
 'etag': 'rkRIPP44rbbH8VVDAY4kMvpf_3I',
 'nextPageToken': 'Z2V0X3JhbmtlZF9zdHJlYW1zLS1Da1VJZ0FRVkY3ZlJPQm83Q2pjSTJGOFFnQVFZQnlJdGwyMDBMV3BlVVVlemlZYWlMTmlDNVJOSi1YVEdFQU5NbkJvMTdsRFZqWGVHZ1h5TEtNdFJnNEZ3WkFBQkVBVVNCUWlKSUJnQUVnVUlxQ0FZQUJJRkNJZ2dHQUFTQlFpSElCZ0FFZ2NJaFNBUUFSZ0JFZ2NJaENBUUJSZ0JFZ1VJdXlBWUFB',
 'pageInfo': {'totalResults': 5, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#commentThread',
   'etag': 'Dzq8Z0vrZgj9MUOKcaf5qwAnJBw',
   'id': 'UgxjKWnCbTmpKSVY6xB4AaABAg',
   'snippet': {'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
    'videoId': 'pXRepg6uSSc',
    'topLevelComment': {'kind': 'youtube#comment',
     'etag': 'FFJJAjakihgAImdHXKX8VXbsLR0',
     'id': 'UgxjKWnCbTmpKSVY6xB4AaABAg',
     'snippet': {'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
      'videoId': 'pXRepg6uSSc',
      'textDisplay': 'bro randomly woke up and decided to do ts 😭✌️',
      'textOriginal': 'bro randomly woke up and decided to do ts 😭✌️',
      'aut

In [20]:
# Print The Top Comments
for item in comment_response.get('items', []):
    comment_text = item['snippet']['topLevelComment']['snippet']['textDisplay']
    # Cutting off at 100 characters for clean printing
    print(f"- {comment_text[:100]}...")

- bro randomly woke up and decided to do ts 😭✌️...
- Just I.N being I.N, he's so cute 😭...
- This is why they say maknae on top...
- "Jeong-in, it's time to go to bed" 😔✨...
- Yall don’t understand how funny this is at almost midnight with no volume lmaoo 😭...


In [21]:
# Put it into a DataFrame
# If we had run a loop over all videos, we'd have a list of dictionaries like this:
mock_video_data = [{
    'video_id': video_ids[0],
    'title': title,
    'content_pillar': categorize_content(title),
    'view_count': int(stats.get('viewCount', 0))
}]

df_videos = pd.DataFrame(mock_video_data)
display(df_videos)

,video_id,title,content_pillar,view_count
0,pXRepg6uSSc,빵발란스한 #StrayKids #스트레이키즈 #아이엔 #I_N #YouMakeStr...,Other/Misc,899261
